<a href="https://colab.research.google.com/github/Ans-butt/ObjectDetection/blob/main/Project_Cleaned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO26 Flask Dashboard — Clean Setup

Run the cells **from top to bottom, once each**. Do not run an old Flask or Cloudflare cell afterward, because restarting the tunnel changes the temporary URL.

The login/signup screen is a frontend demo only; it does not authenticate users securely.

In [10]:
%pip install -q ultralytics flask requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 6.5 MB/s eta 0:00:00


## 1. Check runtime and model

Enable a GPU in **Runtime → Change runtime type → T4 GPU** if available. The app can run on CPU too, but more slowly.

In [1]:
import os, torch

MODEL_PATH = "/content/yolo26s.pt"
print("Python environment ready.")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if not os.path.exists(MODEL_PATH):
    from google.colab import files
    print("Upload yolo26s.pt")
    uploaded = files.upload()
    if not os.path.exists(MODEL_PATH):
        # Rename a single uploaded .pt model to the expected path.
        pt_files = [name for name in uploaded if name.lower().endswith(".pt")]
        if len(pt_files) == 1:
            os.replace(pt_files[0], MODEL_PATH)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Model not found. Upload yolo26s.pt.")
print("Model found:", MODEL_PATH)


Python environment ready.
CUDA available: False
Upload yolo26s.pt


Saving yolo26s.pt to yolo26s.pt
Model found: /content/yolo26s.pt


## 2. Write the Flask backend

This writes `/content/app.py` before starting the server.

In [2]:
%%writefile /content/app.py
from flask import Flask, render_template, Response, jsonify, request, send_from_directory
from ultralytics import YOLO
import torch
import cv2
import os
import time
import threading
import numpy as np
from datetime import datetime

app = Flask(__name__)

BASE_DIR = "/content"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
TEMPLATE_DIR = os.path.join(BASE_DIR, "templates")

os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(TEMPLATE_DIR, exist_ok=True)

MODEL_PATH = "/content/yolo26s.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 416
CONFIDENCE = 0.5
JPEG_QUALITY = 75

# Object must be absent for this long to become eligible again.
OBJECT_ABSENCE_SECONDS = 3.0

# Global limit: save no more than one image every 10 seconds.
MIN_SECONDS_BETWEEN_SAVES = 10.0
last_image_save_time = 0.0

# Load the model once.
model = YOLO(
    MODEL_PATH if os.path.exists(MODEL_PATH) else "yolo26s.pt"
)

inference_lock = threading.Lock()
state_lock = threading.Lock()

camera_active = False

# Tracking state
last_seen_by_track = {}
captured_tracks = set()

stats = {
    "active": False,
    "total_frames": 0,
    "objects_detected": 0,
    "inference_ms": 0.0,
    "fps": 0.0,
    "saved_images": []
}


@app.route("/")
def index():
    return render_template("index.html")


@app.route("/start-camera", methods=["POST"])
def start_camera():
    global camera_active

    with inference_lock:
        with state_lock:
            camera_active = True
            stats["active"] = True

        last_seen_by_track.clear()
        captured_tracks.clear()

    return jsonify({
        "success": True,
        "message": "Detection started."
    })


@app.route("/stop-camera", methods=["POST"])
def stop_camera():
    global camera_active

    with state_lock:
        camera_active = False
        stats["active"] = False

    return jsonify({
        "success": True,
        "message": "Detection stopped."
    })


@app.route("/camera-status")
def camera_status():
    with state_lock:
        return jsonify(dict(stats))


@app.route("/process-frame", methods=["POST"])
def process_frame():
    global last_image_save_time

    with state_lock:
        if not camera_active:
            return jsonify({
                "success": False,
                "message": "Start detection first."
            }), 409

    uploaded_frame = request.files.get("frame")

    if uploaded_frame is None:
        return jsonify({
            "success": False,
            "message": "No frame received."
        }), 400

    image_bytes = uploaded_frame.read()
    frame_array = np.frombuffer(image_bytes, dtype=np.uint8)
    frame = cv2.imdecode(frame_array, cv2.IMREAD_COLOR)

    if frame is None:
        return jsonify({
            "success": False,
            "message": "Invalid image frame."
        }), 400

    start_time = time.perf_counter()

    try:
        # Keep tracking and image-saving operations synchronized.
        with inference_lock:
            with state_lock:
                if not camera_active:
                    return jsonify({
                        "success": False,
                        "message": "Detection stopped."
                    }), 409

            results = model.track(
                source=frame,
                persist=True,
                tracker="bytetrack.yaml",
                conf=CONFIDENCE,
                imgsz=IMAGE_SIZE,
                device=DEVICE,
                verbose=False
            )

            result = results[0]
            annotated_frame = result.plot()

            object_count = (
                len(result.boxes)
                if result.boxes is not None
                else 0
            )

            elapsed = time.perf_counter() - start_time
            inference_ms = elapsed * 1000
            effective_fps = (
                1.0 / elapsed if elapsed > 0 else 0.0
            )

            with state_lock:
                stats["total_frames"] += 1
                stats["objects_detected"] = object_count
                stats["inference_ms"] = round(inference_ms, 2)
                stats["fps"] = round(effective_fps, 2)

            # ---------------------------------------
            # OBJECT TRACKING AND IMAGE SAVING
            # ---------------------------------------
            now = time.monotonic()

            tracked_ids = []

            if (
                result.boxes is not None
                and result.boxes.id is not None
            ):
                tracked_ids = (
                    result.boxes.id.int().cpu().tolist()
                )

            # Release IDs that have been absent for 3 seconds.
            expired_tracks = [
                track_id
                for track_id, last_seen
                in last_seen_by_track.items()
                if now - last_seen >= OBJECT_ABSENCE_SECONDS
            ]

            for track_id in expired_tracks:
                last_seen_by_track.pop(track_id, None)
                captured_tracks.discard(track_id)

            # Find IDs that have not already triggered a capture.
            new_tracks = [
                track_id
                for track_id in tracked_ids
                if track_id not in captured_tracks
            ]

            # Save at most one image every 10 seconds.
            can_save = (
                now - last_image_save_time
                >= MIN_SECONDS_BETWEEN_SAVES
            )

            if new_tracks and can_save:
                filename = datetime.now().strftime(
                    "detection_%Y%m%d_%H%M%S_%f.jpg"
                )
                save_path = os.path.join(IMAGE_DIR, filename)

                if cv2.imwrite(save_path, annotated_frame):
                    last_image_save_time = now

                    # Mark IDs captured only after successful saving.
                    captured_tracks.update(new_tracks)

                    with state_lock:
                        stats["saved_images"].insert(0, filename)
                        stats["saved_images"] = (
                            stats["saved_images"][:50]
                        )

                    print(
                        f"Saved image: {filename} | "
                        f"Tracking IDs: {new_tracks}"
                    )

            # Update the last-seen timestamp for visible objects.
            for track_id in tracked_ids:
                last_seen_by_track[track_id] = now

        # Return the annotated frame to the browser.
        success, encoded = cv2.imencode(
            ".jpg",
            annotated_frame,
            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
        )

        if not success:
            return jsonify({
                "success": False,
                "message": "Could not encode output frame."
            }), 500

        response = Response(
            encoded.tobytes(),
            mimetype="image/jpeg"
        )

        response.headers["X-Detection-Count"] = str(object_count)
        response.headers["X-Inference-ms"] = str(
            round(inference_ms, 2)
        )
        response.headers["Cache-Control"] = "no-store"

        return response

    except Exception:
        app.logger.exception("YOLO inference failed")
        return jsonify({
            "success": False,
            "message": "Inference failed. Check the Colab logs."
        }), 500


@app.route("/saved-images")
def saved_images():
    with state_lock:
        return jsonify(stats["saved_images"])


@app.route("/images/<path:filename>")
def serve_image(filename):
    return send_from_directory(IMAGE_DIR, filename)


if __name__ == "__main__":
    print("Starting YOLO26 dashboard...")
    print("Inference device:", DEVICE)
    print(
        "Minimum seconds between saved images:",
        MIN_SECONDS_BETWEEN_SAVES
    )
    print(
        "Object absence threshold:",
        OBJECT_ABSENCE_SECONDS
    )

    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        threaded=True,
        use_reloader=False
    )


Writing /content/app.py


## 3. Write the dashboard frontend

In [8]:
import os
os.makedirs("/content/templates", exist_ok=True)

with open("/content/templates/index.html", "w", encoding="utf-8") as f:
    f.write("""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>YOLO26 Detection Dashboard</title>

<style>
* { box-sizing: border-box; }

body {
    margin: 0;
    font-family: Arial, sans-serif;
    background: #101827;
    color: #f3f4f6;
}

.hidden { display: none !important; }

.auth-page {
    min-height: 100vh;
    display: flex;
    justify-content: center;
    align-items: center;
    padding: 20px;
    background: linear-gradient(135deg, #101827, #172b46);
}

.auth-card {
    width: 100%;
    max-width: 420px;
    background: #1b2638;
    padding: 32px;
    border-radius: 16px;
    box-shadow: 0 12px 40px #0005;
}

.logo {
    font-size: 30px;
    text-align: center;
    margin-bottom: 8px;
}

h1, h2 { margin-top: 0; }

.auth-card h1 {
    text-align: center;
    font-size: 25px;
}

.subtitle {
    text-align: center;
    color: #aebbd0;
    margin-bottom: 28px;
}

label {
    display: block;
    margin: 16px 0 7px;
    font-size: 14px;
}

input {
    width: 100%;
    padding: 13px;
    border: 1px solid #43516a;
    border-radius: 8px;
    background: #101827;
    color: white;
    font-size: 15px;
}

button {
    padding: 12px 18px;
    border: 0;
    border-radius: 8px;
    cursor: pointer;
    font-weight: bold;
    font-size: 14px;
}

button:disabled {
    opacity: 0.5;
    cursor: not-allowed;
}

.primary {
    width: 100%;
    margin-top: 22px;
    background: #38bdf8;
    color: #082f49;
}

.switch {
    text-align: center;
    margin-top: 22px;
    color: #cbd5e1;
    font-size: 14px;
}

.link {
    color: #38bdf8;
    cursor: pointer;
    font-weight: bold;
}

.notice {
    font-size: 12px;
    color: #fbbf24;
    line-height: 1.5;
    margin-top: 20px;
}

.message {
    font-size: 13px;
    margin-top: 12px;
    overflow-wrap: anywhere;
}

#dashboard {
    max-width: 1100px;
    margin: auto;
    padding: 24px;
}

.topbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 12px;
    flex-wrap: wrap;
}

.muted { color: #aebbd0; }

.panel {
    background: #1b2638;
    padding: 20px;
    margin-top: 20px;
    border-radius: 12px;
}

.buttons {
    display: flex;
    gap: 12px;
    flex-wrap: wrap;
    margin: 16px 0;
}

#startBtn { background: #22c55e; color: #052e16; }
#stopBtn { background: #ef4444; color: white; }
#logoutBtn { background: #334155; color: white; }

#videoOutput {
    display: block;
    width: 100%;
    max-width: 800px;
    min-height: 220px;
    margin: 16px auto;
    object-fit: contain;
    background: #080d16;
    border-radius: 10px;
}

#cameraVideo, #captureCanvas { display: none; }

.stats {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
    gap: 12px;
}

.stat {
    padding: 16px;
    border-radius: 8px;
    background: #26354b;
}

.value {
    font-size: 24px;
    font-weight: bold;
    margin-top: 8px;
}

#gallery {
    display: grid;
    grid-template-columns: repeat(auto-fill, minmax(150px, 1fr));
    gap: 12px;
}

#gallery img {
    width: 100%;
    border-radius: 8px;
}

@media (max-width: 600px) {
    #dashboard { padding: 14px; }
    .auth-card { padding: 24px; }
}
</style>
</head>

<body>

<!-- LOGIN PAGE -->
<section id="loginPage" class="auth-page">
    <div class="auth-card">
        <div class="logo">🎯</div>
        <h1>Welcome Back</h1>
        <p class="subtitle">Sign in to YOLO26 Detection</p>

        <form id="loginForm">
            <label for="loginUsername">Username</label>
            <input id="loginUsername" type="text"
                   placeholder="Enter your username"
                   autocomplete="username" required>

            <label for="loginPassword">Password</label>
            <input id="loginPassword" type="password"
                   placeholder="Enter your password"
                   autocomplete="current-password" required>

            <button class="primary" type="submit">Login</button>
            <p id="loginMessage" class="message" role="status"></p>
        </form>

        <p class="switch">
            Don't have an account?
            <span class="link" id="showSignup">Create account</span>
        </p>


    </div>
</section>

<!-- SIGNUP PAGE -->
<section id="signupPage" class="auth-page hidden">
    <div class="auth-card">
        <div class="logo">🚀</div>
        <h1>Create Account</h1>
        <p class="subtitle">Get started with object detection</p>

        <form id="signupForm">
            <label for="signupUsername">Username</label>
            <input id="signupUsername" type="text"
                   placeholder="Choose a username"
                   autocomplete="username" required>

            <label for="signupPassword">Password</label>
            <input id="signupPassword" type="password"
                   placeholder="Choose a password"
                   autocomplete="new-password" required>

            <label for="confirmPassword">Confirm Password</label>
            <input id="confirmPassword" type="password"
                   placeholder="Enter password again"
                   autocomplete="new-password" required>

            <button class="primary" type="submit">Sign Up</button>
            <p id="signupMessage" class="message" role="status"></p>
        </form>

        <p class="switch">
            Already have an account?
            <span class="link" id="showLogin">Login</span>
        </p>

        <p class="notice">
            This demo does not create a real server account. Any non-empty
            username and password can be used at login.
        </p>
    </div>
</section>

<!-- YOLO DASHBOARD -->
<main id="dashboard" class="hidden">
    <div class="topbar">
        <div>
            <h1>YOLO26 Object Detection</h1>
            <p class="muted">
                GPU-powered live detection dashboard
            </p>
            <p class="muted">
                Signed in as <strong id="currentUser"></strong>
            </p>
        </div>
        <button id="logoutBtn">Logout</button>
    </div>

    <section class="panel">
        <h2>Live Camera</h2>
        <p id="status" role="status">Camera stopped.</p>

        <div class="buttons">
            <button id="startBtn">Start Camera</button>
            <button id="stopBtn">Stop Camera</button>
        </div>

        <video id="cameraVideo" autoplay playsinline muted></video>
        <canvas id="captureCanvas"></canvas>

        <img id="videoOutput" alt="Annotated detection frames">

        <p class="muted">
            Allow webcam access when your browser asks.
        </p>
    </section>

    <section class="panel">
        <h2>Detection Statistics</h2>
        <div class="stats">
            <div class="stat">
                Frames processed
                <div class="value" id="totalFrames">0</div>
            </div>
            <div class="stat">
                Objects detected
                <div class="value" id="objectCount">0</div>
            </div>
            <div class="stat">
                Inference time
                <div class="value" id="inferenceTime">0 ms</div>
            </div>
            <div class="stat">
                Processing FPS
                <div class="value" id="processingFps">0</div>
            </div>
        </div>
    </section>

    <section class="panel">
        <h2>Saved Detection Images</h2>
        <div id="gallery"></div>
    </section>
</main>

<script>
const loginPage = document.getElementById("loginPage");
const signupPage = document.getElementById("signupPage");
const dashboard = document.getElementById("dashboard");

const video = document.getElementById("cameraVideo");
const canvas = document.getElementById("captureCanvas");
const output = document.getElementById("videoOutput");
const context = canvas.getContext("2d");

const statusText = document.getElementById("status");
const startBtn = document.getElementById("startBtn");
const stopBtn = document.getElementById("stopBtn");

let mediaStream = null;
let running = false;
let busy = false;
let lastObjectUrl = null;
let pageSession = false;
let statsTimer = null;
let galleryTimer = null;

function showPage(page) {
    loginPage.classList.toggle("hidden", page !== "login");
    signupPage.classList.toggle("hidden", page !== "signup");
    dashboard.classList.toggle("hidden", page !== "dashboard");
}

function showLoginMessage(message) {
    document.getElementById("loginMessage").textContent = message;
}

function showSignupMessage(message) {
    document.getElementById("signupMessage").textContent = message;
}

document.getElementById("showSignup").addEventListener("click", () => {
    showSignupMessage("");
    showPage("signup");
});

document.getElementById("showLogin").addEventListener("click", () => {
    showLoginMessage("");
    showPage("login");
});

// Demo signup: no credentials are saved or sent to a server.
document.getElementById("signupForm").addEventListener("submit", event => {
    event.preventDefault();

    const username = document.getElementById("signupUsername").value.trim();
    const password = document.getElementById("signupPassword").value;
    const confirm = document.getElementById("confirmPassword").value;

    if (!username || !password) {
        showSignupMessage("Enter a username and password.");
        return;
    }

    if (password !== confirm) {
        showSignupMessage("Passwords do not match.");
        return;
    }

    document.getElementById("loginUsername").value = username;
    document.getElementById("loginPassword").value = "";
    showLoginMessage("Demo signup complete. You can now log in.");
    showPage("login");
});

// Demo login: accepts any non-empty username and password.
document.getElementById("loginForm").addEventListener("submit", event => {
    event.preventDefault();

    const username = document.getElementById("loginUsername").value.trim();
    const password = document.getElementById("loginPassword").value;

    if (!username || !password) {
        showLoginMessage("Enter both username and password.");
        return;
    }

    pageSession = true;
    document.getElementById("currentUser").textContent = username;
    showLoginMessage("");
    showPage("dashboard");

    refreshStats();
    refreshGallery();

    if (statsTimer) clearInterval(statsTimer);
    if (galleryTimer) clearInterval(galleryTimer);

    statsTimer = setInterval(refreshStats, 1500);
    galleryTimer = setInterval(refreshGallery, 5000);
});

function stopLocalCamera() {
    if (mediaStream) {
        mediaStream.getTracks().forEach(track => track.stop());
        mediaStream = null;
    }

    video.srcObject = null;
}

async function stopCamera() {
    running = false;
    startBtn.disabled = false;

    try {
        await fetch("/stop-camera", { method: "POST" });
    } catch (error) {
        console.error("Stop request failed:", error);
    }

    stopLocalCamera();
    statusText.textContent = "Camera stopped.";
}

async function logout() {
    await stopCamera();

    pageSession = false;
    showPage("login");

    if (statsTimer) clearInterval(statsTimer);
    if (galleryTimer) clearInterval(galleryTimer);

    statsTimer = null;
    galleryTimer = null;
}

document.getElementById("logoutBtn").addEventListener("click", logout);

async function startCamera() {
    if (!pageSession || running) return;

    try {
        startBtn.disabled = true;
        statusText.textContent = "Requesting webcam permission...";

        mediaStream = await navigator.mediaDevices.getUserMedia({
            video: {
                width: { ideal: 640 },
                height: { ideal: 480 },
                frameRate: { ideal: 20 }
            },
            audio: false
        });

        video.srcObject = mediaStream;
        await video.play();

        const response = await fetch("/start-camera", {
            method: "POST"
        });

        if (!response.ok) {
            throw new Error("Could not start server-side detection.");
        }

        canvas.width = 640;
        canvas.height = 480;

        running = true;
        statusText.textContent = "Camera running. Sending frames to Colab.";
        captureLoop();

    } catch (error) {
        statusText.textContent = "Camera error: " + error.message;
        stopLocalCamera();
        startBtn.disabled = false;
    }
}

function captureBlob() {
    return new Promise(resolve => {
        canvas.toBlob(resolve, "image/jpeg", 0.70);
    });
}

async function captureLoop() {
    while (running) {
        if (busy || video.readyState < 2) {
            await new Promise(resolve => setTimeout(resolve, 20));
            continue;
        }

        busy = true;

        try {
            context.drawImage(video, 0, 0, canvas.width, canvas.height);
            const blob = await captureBlob();

            if (!blob || !running) continue;

            const formData = new FormData();
            formData.append("frame", blob, "frame.jpg");

            const response = await fetch("/process-frame", {
                method: "POST",
                body: formData
            });

            if (!response.ok) {
                throw new Error("Server returned HTTP " + response.status);
            }

            const imageBlob = await response.blob();

            if (running) {
                const newUrl = URL.createObjectURL(imageBlob);
                output.src = newUrl;

                if (lastObjectUrl) URL.revokeObjectURL(lastObjectUrl);
                lastObjectUrl = newUrl;

                document.getElementById("objectCount").textContent =
                    response.headers.get("X-Detection-Count") || "0";

                document.getElementById("inferenceTime").textContent =
                    (response.headers.get("X-Inference-ms") || "0") + " ms";
            }

        } catch (error) {
            if (running) {
                statusText.textContent = "Streaming error: " + error.message;
            }
            await new Promise(resolve => setTimeout(resolve, 300));
        } finally {
            busy = false;
        }

        await new Promise(resolve => setTimeout(resolve, 20));
    }
}

async function refreshStats() {
    if (!pageSession) return;

    try {
        const response = await fetch("/camera-status", {
            cache: "no-store"
        });

        const data = await response.json();

        document.getElementById("totalFrames").textContent =
            data.total_frames ?? 0;

        document.getElementById("processingFps").textContent =
            data.fps ?? 0;
    } catch (error) {
        console.error("Stats update failed:", error);
    }
}

async function refreshGallery() {
    if (!pageSession) return;

    try {
        const response = await fetch("/saved-images", {
            cache: "no-store"
        });

        const images = await response.json();
        const gallery = document.getElementById("gallery");

        gallery.replaceChildren();

        images.forEach(filename => {
            const image = document.createElement("img");
            image.loading = "lazy";
            image.alt = "Saved detection";
            image.src = "/images/" + encodeURIComponent(filename);
            gallery.appendChild(image);
        });
    } catch (error) {
        console.error("Gallery update failed:", error);
    }
}

startBtn.addEventListener("click", startCamera);
stopBtn.addEventListener("click", stopCamera);

// Always require the demo login after opening or refreshing the page.
showPage("login");
</script>

</body>
</html>""")

## 4. Start Flask once and verify it

Keep this notebook cell running only long enough to confirm startup; the Flask process itself runs in the background.

In [11]:
import os, subprocess, time, requests

# Stop only the Flask process launched by this notebook, if rerunning.
if "app_process" in globals() and app_process.poll() is None:
    app_process.terminate()
    try:
        app_process.wait(timeout=8)
    except subprocess.TimeoutExpired:
        app_process.kill()
        app_process.wait()

flask_log = open("/content/flask.log", "w")
app_process = subprocess.Popen(
    ["python", "-u", "/content/app.py"],
    cwd="/content",
    stdout=flask_log,
    stderr=subprocess.STDOUT
)

flask_ok = False
for _ in range(30):
    time.sleep(1)
    if app_process.poll() is not None:
        break
    try:
        response = requests.get("http://127.0.0.1:5000/", timeout=2)
        if response.status_code == 200:
            flask_ok = True
            break
    except requests.RequestException:
        pass

if flask_ok:
    print("Flask is responding:", response.status_code)
    print("Login page found:", 'id="loginPage"' in response.text)
    print("Signup page found:", 'id="signupPage"' in response.text)
else:
    print("Flask failed to start. Log output:")
    print(open("/content/flask.log").read()[-6000:])
    raise RuntimeError("Fix the Flask error shown above before starting Cloudflare.")


Flask is responding: 200
Login page found: True
Signup page found: True


## 5. Start one fresh Cloudflare tunnel

Use only the URL printed by this cell. Quick Tunnel URLs are temporary; if you restart the tunnel, use the newly printed URL.

In [12]:
import os, re, time, subprocess, requests

binary_path = "/content/cloudflared"
log_path = "/content/cloudflared.log"

# Stop a tunnel previously launched by this notebook.
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=8)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()
        tunnel_process.wait()

if not os.path.exists(binary_path):
    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/"
        "download/cloudflared-linux-amd64"
    )
    print("Downloading Cloudflare Tunnel...")
    response = requests.get(download_url, timeout=90)
    response.raise_for_status()
    with open(binary_path, "wb") as f:
        f.write(response.content)
    os.chmod(binary_path, 0o755)

with open(log_path, "w") as log_file:
    tunnel_process = subprocess.Popen(
        [
            binary_path, "tunnel", "--no-autoupdate",
            "--protocol", "http2",
            "--url", "http://127.0.0.1:5000"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

dashboard_url = None
for _ in range(45):
    time.sleep(2)
    if tunnel_process.poll() is not None:
        break
    try:
        with open(log_path, "r") as f:
            logs = f.read()
    except OSError:
        logs = ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)
    if match:
        dashboard_url = match.group(0)
        break

if dashboard_url:
    print("OPEN THIS DASHBOARD URL:", dashboard_url)
else:
    print("No tunnel URL was found. Recent logs:")
    print(open(log_path).read()[-6000:])
    raise RuntimeError("Cloudflare tunnel did not provide a URL.")


OPEN THIS DASHBOARD URL: https://attachments-york-drivers-connections.trycloudflare.com


## Troubleshooting

- If Flask fails, inspect `/content/flask.log` and fix that error before launching Cloudflare.
- If the URL stops working, rerun only the Cloudflare cell and use its **new** URL.
- Keep the Colab runtime connected and do not stop the Flask/tunnel processes while using the dashboard.